In [47]:
import math

In [55]:
class Value:
    def __init__(self,data,_child = (),op = '',grad = 0.0):
        self.data = data
        self._prev = set(_child)
        self.op = op
        self.grad = 0.0
        self._backward = lambda : None
    def __add__(self,other):
        out = Value(self.data + other.data,(self,other),'+')
        def _backward(): 
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0
        out._backward = _backward
        return out
    def __mul__(self,other):
        out = Value(self.data * other.data,(self,other),'*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def exp(self,other):
        out = Value(math.exp(self.data),(self,other),'exp')
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for child in reversed(topo):
            child._backward()
    def __pow__(self,other):
        data = other.data
        out = Value(self.data * math.pow(self.data,data),(self,other),'**')
    def __truediv__(self,other):
        return self * (other ** -1)
    def __radd__(self,other):
        return self + other
    def __rmul__(self,other):
        out = other * self
    def __repr__(self):
        return f'Value : {self.data} , operation : {self.op}'
    

In [56]:
a = Value(2.0)
b = Value(3.0)
c = a + b
c = a * b + b
c.backward()
print(a.grad,b.grad,c.grad)

3.0 3.0 1.0
